# Fine-tune Gemma 4 E4B — NT housing maintenance triage parser

Trains the parser on a free Colab GPU, evaluates it against the base model, merges
the LoRA adapter and exports a **Q4_K_M GGUF** for Ollama. Nothing heavy is
downloaded to your laptop — only the finished model comes back.

**Setup**

1. `Runtime -> Change runtime type -> T4 GPU` (free tier).
2. Run cells top to bottom. When the *Load dataset* cell prompts, upload
   `train.jsonl` **and** `eval.jsonl` from `nt-housing-triage/training/out/`.
3. Keep the tab open — free Colab disconnects when idle.
4. If cell 4 fails to import `unsloth`, do `Runtime -> Restart session` and re-run
   from cell 2 (Colab sometimes needs a restart right after installing Unsloth).

**Outputs**

- `gemma4-e4b-lora.zip` — the LoRA adapter (tens of MB)
- `unsloth.Q4_K_M.gguf` — the model (~5 GB)

**Then locally** (in `nt-housing-triage/`):

```
# drop the GGUF at training/models/gguf/unsloth.Q4_K_M.gguf
ollama create nt-housing-triage -f training/Modelfile
```

In [ ]:
#@title 1. Check GPU + install Unsloth { display-mode: "form" }
!nvidia-smi -L
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "| bf16", torch.cuda.is_bf16_supported())
!pip install -q unsloth

In [ ]:
#@title 2. Config { display-mode: "form" }
MODEL          = "unsloth/gemma-4-E4B-it"  # same model as the local GGUF
MAX_SEQ_LENGTH = 512                        # reports are short; keeps VRAM low
EPOCHS         = 2
BATCH_SIZE     = 2
GRAD_ACCUM     = 4
LR             = 2e-4
LORA_R         = 16
LORA_ALPHA     = 16
CHAT_TEMPLATE  = "gemma-4"                  # non-thinking template for E2B/E4B

In [ ]:
#@title 3. Load dataset (upload train.jsonl + eval.jsonl) { display-mode: "form" }
import os
from google.colab import files

if not (os.path.exists("train.jsonl") and os.path.exists("eval.jsonl")):
    print("Select BOTH train.jsonl and eval.jsonl from nt-housing-triage/training/out/")
    files.upload()
print("found:", [f for f in os.listdir(".") if f.endswith(".jsonl")])

In [ ]:
#@title 4. Load Gemma 4 E4B in 4-bit + attach LoRA { display-mode: "form" }
from unsloth import FastModel

model, tokenizer = FastModel.from_pretrained(
    model_name=MODEL,
    dtype=None,               # auto (fp16 on the T4)
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,        # QLoRA
    full_finetuning=False,
)
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers=False,    # text-only task
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0,
    bias="none",
    random_state=3407,
)

In [ ]:
#@title 5. Apply the Gemma 4 chat template { display-mode: "form" }
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(tokenizer, chat_template=CHAT_TEMPLATE)

from datasets import load_dataset
ds = load_dataset("json", data_files={"train": "train.jsonl", "eval": "eval.jsonl"})

def format_batch(batch):
    return {
        "text": [
            tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=False).removeprefix("<bos>")
            for m in batch["messages"]
        ]
    }

ds = ds.map(format_batch, batched=True, remove_columns=ds["train"].column_names)
print(ds)
print(ds["train"][0]["text"][:400])

In [ ]:
#@title 6. Train (QLoRA) { display-mode: "form" }
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=ds["train"],
    eval_dataset=ds["eval"],
    args=SFTConfig(
        dataset_text_field="text",
        max_length=MAX_SEQ_LENGTH,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        warmup_ratio=0.03,
        num_train_epochs=EPOCHS,
        learning_rate=LR,
        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.001,
        lr_scheduler_type="linear",
        seed=3407,
        report_to="none",
        output_dir="outputs",
        eval_strategy="steps",
        eval_steps=100,
        save_steps=100,
        save_total_limit=2,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
    ),
)
# Only train on the assistant JSON, not the system prompt / report.
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|turn>user\n",
    response_part="<|turn>model\n",
)
trainer.train()

In [ ]:
#@title 7. Save + download the adapter { display-mode: "form" }
model.save_pretrained("gemma4-e4b-lora")
tokenizer.save_pretrained("gemma4-e4b-lora")
!zip -qr gemma4-e4b-lora.zip gemma4-e4b-lora
from google.colab import files
files.download("gemma4-e4b-lora.zip")

In [ ]:
#@title 8. Evaluate: field-level F1 on the held-out split { display-mode: "form" }
import json
from collections import defaultdict

SINGLE = ("category", "safety_level", "trade_required", "community")
SETS = ("urgency_flags", "occupant_vulnerability")

def extract_json(text):
    s, e = text.find("{"), text.rfind("}")
    if s == -1 or e <= s:
        return None
    try:
        return json.loads(text[s:e + 1])
    except Exception:
        return None

def set_f1(pred, gold):
    p, g = set(pred or []), set(gold or [])
    tp = len(p & g)
    prec = tp / len(p) if p else (1.0 if not g else 0.0)
    rec = tp / len(g) if g else 1.0
    return 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0

hits, total, f1s = defaultdict(int), defaultdict(int), defaultdict(list)
valid = 0
rows = [json.loads(l) for l in open("eval.jsonl") if l.strip()]
for i, row in enumerate(rows):
    report = next(m["content"] for m in row["messages"] if m["role"] == "user")
    gold = json.loads(next(m["content"] for m in row["messages"] if m["role"] == "assistant"))
    inputs = tokenizer.apply_chat_template(
        [{"role": "user", "content": report}],
        add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt",
    ).to("cuda")
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=192, do_sample=False, use_cache=True)
    pred = extract_json(tokenizer.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True))
    if pred:
        valid += 1
        for f in SINGLE:
            total[f] += 1
            if pred.get(f) == gold.get(f):
                hits[f] += 1
        for f in SETS:
            f1s[f].append(set_f1(pred.get(f), gold.get(f)))
    if (i + 1) % 50 == 0:
        print(f"{i + 1}/{len(rows)}")

print(f"\njson_valid: {valid / len(rows):.3f}")
for f in SINGLE:
    print(f"{f:<24}: {hits[f] / total[f]:.3f}" if total[f] else f"{f:<24}: n/a")
for f in SETS:
    print(f"{f:<24}: F1 {sum(f1s[f]) / len(f1s[f]):.3f}" if f1s[f] else f"{f:<24}: n/a")

In [ ]:
#@title 9. Merge + export Q4_K_M GGUF { display-mode: "form" }
model.save_pretrained_gguf("gguf", tokenizer, quantization_method="q4_k_m")
!ls -lh gguf

In [ ]:
#@title 10. Download the GGUF (or push to Hugging Face Hub) { display-mode: "form" }
# Option A: direct download (~5 GB). Can drop on flaky connections.
from google.colab import files
files.download("gguf/unsloth.Q4_K_M.gguf")

# Option B (more reliable): push to the Hub, then pull it down from there.
# from huggingface_hub import login
# login()  # paste a WRITE token
# model.push_to_hub_gguf("YOUR_USER/nt-housing-triage-gguf", tokenizer, quantization_method="q4_k_m")